# Mission EdStats : analyser les indicateurs retenus et proposer une liste de pays (phase 3)

La phase précédente a réduit le tableau pays × indicateurs aux seuls indicateurs faiblement corrélés entre eux (`country_indicator_reduced.csv` dans `data/output/`).

**Objectif** : analyser chaque indicateur restant par pays et formuler une liste de pays pertinents pour l'implantation d'academy, en réponse à la demande de Mark.

Plan de ce notebook : présenter le tableau réduit (section 1), étudier la distribution de chaque indicateur (section 2), proposer une méthode quantitative de sélection des pays (section 3), conclure par la liste de pays proposée (section 4).

## 1. Présenter le tableau réduit

L'entrée de cette analyse est le CSV écrit par la phase précédente dans `data/output/` : `country_indicator_reduced.csv` (tableau pays × indicateurs réduits) et `EdStatsSeries.csv` (nom complet de chaque code d'indicateur).

### 1.1 Dossiers et lecture

`load_csv_output()` lit un CSV de `data/output/`, que le notebook soit lancé depuis la racine du projet ou depuis `notebooks/`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_colwidth", None)  # show full indicator descriptions

candidates = [
    Path.cwd() / "data" / "output",         # from the project root
    Path.cwd() / ".." / "data" / "output",  # from notebooks/
]

OUTPUT_DIR = None
for candidate in candidates:
    if candidate.exists():
        OUTPUT_DIR = candidate.resolve()
        break
if OUTPUT_DIR is None:
    raise FileNotFoundError("data/output/ not found")

def load_csv_output(name, columns=None):
    """Read the CSV `name` from OUTPUT_DIR as is, optionally keeping only `columns`."""
    return pd.read_csv(OUTPUT_DIR / name, encoding="utf-8", usecols=columns)

OUTPUT_DIR

PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/output')

### 1.2 Le tableau réduit pays × indicateurs

Chargé une seule fois dans `reduced_table` et réutilisé par toutes les cellules suivantes : aucune opération ne modifie ce DataFrame en place, on recharge le CSV si besoin. Aperçu des 3 premières lignes :

In [2]:
# load once, reused by every cell of this notebook
reduced_table = load_csv_output("country_indicator_reduced.csv")
reduced_table.head(10)

,Country Code,Country Name,IT.NET.USER.P2,SP.SEC.UTOT.IN,SP.TER.TOTL.IN,SE.SEC.ENRR.UP,SE.TER.ENRR,PRJ.MYS.1519.MF,NY.GDP.PCAP.CD,SE.SEC.ENRL.TC.ZS,SE.XPD.TOTL.GD.ZS
0,ABW,Aruba,60.292077,4.478191e+03,6.799110e+03,90.496135,28.226781,8.259259,24357.132282,14.201038,5.872939
1,AFG,Afghanistan,4.332797,1.865566e+06,2.624559e+06,32.586328,4.899025,NaN,489.193612,35.668039,3.342328
2,AGO,Angola,5.180553,1.415692e+06,2.006447e+06,17.902200,6.445411,NaN,3483.189042,25.717081,3.100720
3,ALB,Albania,39.532074,1.980868e+05,2.749237e+05,78.346604,44.499039,8.792593,3833.464309,14.913130,3.289472
4,AND,Andorra,75.188177,NaN,NaN,NaN,NaN,NaN,40289.594889,7.972084,2.719720
5,ARE,United Arab Emirates,70.247694,2.328435e+05,3.399750e+05,NaN,NaN,9.540741,39984.323639,NaN,NaN
6,ARG,Argentina,43.902018,2.060006e+06,3.392539e+06,76.756416,73.124107,8.918519,10079.971248,14.431342,4.892109
7,ARM,Armenia,28.524414,1.163443e+05,2.789649e+05,89.769697,44.427286,8.940741,3142.616214,NaN,2.843909
8,ASM,American Samoa,NaN,NaN,NaN,NaN,NaN,NaN,10621.848286,NaN,NaN
9,ATG,Antigua and Barbuda,48.362707,3.069309e+03,7.225493e+03,84.146561,17.486815,NaN,12565.487532,11.937176,2.760067


Taille du tableau (lignes = pays, colonnes = `Country Code`, `Country Name` et les indicateurs) :

In [3]:
print(f"{reduced_table.shape[0]} rows, {reduced_table.shape[1]} columns")

214 rows, 11 columns


### 1.3 Les indicateurs retenus et leur signification

`EdStatsSeries.csv` associe chaque code (ex. `IT.NET.USER.P2`) à son nom officiel en anglais (`Indicator Name`, nom de l'indicateur). Pour lire le tableau plus facilement, chaque code est complété d'une courte description en français, écrite dans le dictionnaire `indicator_descriptions_fr` ci-dessous (modifiable). Tableau complet des indicateurs retenus :

In [4]:
# one short French description per retained indicator code, editable
indicator_descriptions_fr = {
    "IT.NET.USER.P2": "Utilisateurs d'Internet pour 100 habitants (usage au cours des 3 derniers mois)",
    "SP.SEC.UTOT.IN": "Population d'âge théorique du secondaire supérieur (nombre de personnes)",
    "SP.TER.TOTL.IN": "Population d'âge théorique du supérieur (nombre de personnes)",
    "SE.SEC.ENRR.UP": "Taux brut de scolarisation dans le secondaire supérieur (%)",
    "SE.TER.ENRR": "Taux brut de scolarisation dans le supérieur (%)",
    "PRJ.MYS.1519.MF": "Projection du nombre moyen d'années de scolarité des 15-19 ans",
    "NY.GDP.PCAP.CD": "PIB par habitant, en dollars US courants",
    "SE.SEC.ENRL.TC.ZS": "Nombre moyen d'élèves par enseignant dans le secondaire",
    "SE.XPD.TOTL.GD.ZS": "Dépenses publiques d'éducation, en % du PIB",
}

indicator_columns = []
for col in reduced_table.columns:
    if col not in ("Country Code", "Country Name"):
        indicator_columns.append(col)

# load once, reused later for indicator names
series = load_csv_output("EdStatsSeries.csv", columns=["Series Code", "Indicator Name"])
series = series[series["Series Code"].isin(indicator_columns)]

# full table: code, official English name, French description
rows = []
for code in indicator_columns:
    name = series.loc[series["Series Code"] == code, "Indicator Name"].iloc[0]
    rows.append({"Code": code, "Indicator Name": name, "Description (français)": indicator_descriptions_fr[code]})
indicators = pd.DataFrame(rows)
indicators

,Code,Indicator Name,Description (français)
0,IT.NET.USER.P2,Internet users (per 100 people),Utilisateurs d'Internet pour 100 habitants (usage au cours des 3 derniers mois)
1,SP.SEC.UTOT.IN,"Population of the official age for upper secondary education, both sexes (number)",Population d'âge théorique du secondaire supérieur (nombre de personnes)
2,SP.TER.TOTL.IN,"Population of the official age for tertiary education, both sexes (number)",Population d'âge théorique du supérieur (nombre de personnes)
3,SE.SEC.ENRR.UP,"Gross enrolment ratio, upper secondary, both sexes (%)",Taux brut de scolarisation dans le secondaire supérieur (%)
4,SE.TER.ENRR,"Gross enrolment ratio, tertiary, both sexes (%)",Taux brut de scolarisation dans le supérieur (%)
5,PRJ.MYS.1519.MF,Projection: Mean years of schooling. Age 15-19. Total,Projection du nombre moyen d'années de scolarité des 15-19 ans
6,NY.GDP.PCAP.CD,GDP per capita (current US$),"PIB par habitant, en dollars US courants"
7,SE.SEC.ENRL.TC.ZS,Pupil-teacher ratio in secondary education (headcount basis),Nombre moyen d'élèves par enseignant dans le secondaire
8,SE.XPD.TOTL.GD.ZS,Government expenditure on education as % of GDP (%),"Dépenses publiques d'éducation, en % du PIB"


### Bilan de la section 1

Tableau réduit chargé une seule fois depuis `data/output/` (`country_indicator_reduced.csv`), taille affichée, et pour chaque indicateur retenu : nom officiel (`EdStatsSeries.csv`) et description en français.